In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')
import seaborn as sns
from IPython.display import Audio

import sys
sys.path.insert(0, '../scripts')
from paths import DATA_ROOT  # participant data lives outside the repo; see scripts/paths.py


In [ ]:
df = pd.read_csv(DATA_ROOT / 'aggregated_segments_with_embeddings.csv')
embcols = [c for c in df.columns if c.startswith('emb_')]

In [ ]:
df['audio_path'].iloc[0]

In [ ]:
# # compress to 2D using UMAP
# umap = UMAP(n_components=2, random_state=42)
# embeddings_2d = umap.fit_transform(df[embcols].values)  

In [ ]:
# # show the 2D embeddings
# plt.figure(figsize=(10, 8))
# sns.scatterplot(x=embeddings_2d[:, 0], y=embeddings_2d[:, 1], palette='tab10', s=50, alpha=0.7)
# plt.title('2D UMAP Embeddings of Arts Integration Data')    

In [ ]:
# # load the n_neighbors x min_dist grid from scripts/umap_grid.py (rows match df)
# grid = np.load(DATA_ROOT / 'umap_grid.npz')
# grid_emb, n_neighbors, min_dist = grid['embeddings'], grid['n_neighbors'], grid['min_dist']
# grid_emb.shape

In [ ]:
# # 10x10 grid: rows = n_neighbors, columns = min_dist
# nr, nc = len(n_neighbors), len(min_dist)
# fig, axes = plt.subplots(nr, nc, figsize=(2 * nc, 2 * nr))
# for i, nn in enumerate(n_neighbors):
#     for j, md in enumerate(min_dist):
#         ax = axes[i, j]
#         e = grid_emb[i, j]
#         ax.scatter(e[:, 0], e[:, 1], s=1, alpha=0.5, linewidths=0)
#         ax.set_xticks([]); ax.set_yticks([])
#         if i == 0:
#             ax.set_title(f'min_dist={md:g}', fontsize=9)
#         if j == 0:
#             ax.set_ylabel(f'n_neighbors={nn}', fontsize=9)
# fig.suptitle(f"UMAP grid search ({grid['metric']})", y=1.0)
# plt.tight_layout()

# Utterance chooser
Demo: choose a random utterance and then find the closest utterances to it in embedding space. 

In [ ]:
import wave

def load_segment_audio(row, pad=0.1):
    """Read just one segment from its isolated track -> (int16 samples, sample rate).
    row needs audio_path (relative to DATA_ROOT), start and end (seconds into the track); pad adds
    a little either side, since WhisperX's segment boundaries can clip the first/last word."""
    with wave.open(str(DATA_ROOT / row['audio_path'])) as w:
        rate = w.getframerate()
        a = max(0, int((row['start'] - pad) * rate))
        b = min(w.getnframes(), int((row['end'] + pad) * rate))
        w.setpos(a)
        return np.frombuffer(w.readframes(b - a), dtype=np.int16), rate

def concat_audio(rows, gap=0.5):
    """Concatenate segments with `gap` seconds of silence between them -> Audio player."""
    clips = [load_segment_audio(r) for r in rows]
    rate = clips[0][1]
    assert all(r == rate for _, r in clips), 'sample rates differ'
    silence = np.zeros(int(gap * rate), dtype=np.int16)
    parts = [x for a, _ in clips for x in (a, silence)][:-1]
    return Audio(np.concatenate(parts), rate=rate)

def choose_random_utterance(df, k=10):
    """Choose a random utterance from the dataframe and then choose the closest utterances to it in embedding space,
    searching only utterances about the same image from other pairs. Also returns an Audio player of the
    source snippet followed by the closest snippet."""
    # df.iloc[idx] on a mixed-dtype frame gives an object Series, so pull a float matrix instead
    X = df[embcols].to_numpy(np.float64)
    idx = np.random.randint(0, df.shape[0])
    utterance = df.iloc[idx]
    # cosine distance to all other embeddings (same metric as scripts/umap_grid.py)
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    dists = 1 - Xn @ Xn[idx]
    # candidates: same image, different pair (which also excludes the utterance itself)
    candidate = (df['imgID'] == utterance['imgID']) & (df['pairID'] != utterance['pairID'])
    dists[~candidate.to_numpy()] = np.inf
    closest_idx = np.argsort(dists)[:min(k, candidate.sum())]
    closest_utterances = df.iloc[closest_idx].drop(columns=embcols).assign(dist=dists[closest_idx])
    audio = concat_audio([utterance, closest_utterances.iloc[0]])
    return utterance, closest_utterances, audio

In [ ]:
utterance, closest, audio = choose_random_utterance(df)
print(utterance['pairID'], utterance['imgID'], utterance['text'])
print("\nClosest utterance:")
print(closest['text'].iloc[0])
display(audio)  # source snippet -> closest snippet
closest[['pairID', 'imgID', 'text', 'dist']]

# Some notes from the demo (addressed now):
- Some utterances are too long. 
- Audio isolation is not perfect - there are many cases of the non-target's speech being transcribed. 

To address these:
We should re-generate the utterance bank by:
- Applying voicolate's audio isolation to the continuous, concatenated recordings --> transcribe. 
- Separate out utterances either by segment or by individual sentences using punctuation. 

This also means we should have individual tracks be continuously recorded for the whole session, rather than having multiple separate recordings. This will help with the isolation and transcription process.

In [ ]:
# create a cosine distance matrix for all utterances
from sklearn.metrics.pairwise import cosine_distances
X = df[embcols].to_numpy(np.float64)
Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
dist_matrix = cosine_distances(Xn)

In [ ]:
# plot a hist of the lower triangle of the distance matrix (excluding the diagonal)
tril_indices = np.tril_indices(dist_matrix.shape[0], k=-1)
plt.figure(figsize=(8, 6))
sns.histplot(dist_matrix[tril_indices], bins=50);

In [ ]:
## Need to run 2-4 still